In [133]:
import warnings
warnings.filterwarnings('ignore')

In [134]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt

In [135]:
loan_data=pd.read_csv('loan_applications.csv')
transaction_data=pd.read_csv('transaction_history.csv')

In [136]:
loan_copy=loan_data.copy()
transaction_copy=transaction_data.copy()

In [137]:
loan_copy.head()

,application_id,customer_id,application_date,loan_product,loan_purpose,loan_amount_KES,tenure_months,interest_rate_pct,age,gender,...,debt_to_income_ratio,credit_score,num_existing_loans,credit_utilization_pct,num_late_payments_12m,has_collateral,application_status,defaulted,days_past_due,duplicate_applicant
0,LN0000001,CUS002206,2020-04-07 16:07:23,Mortgage,Business Expansion,26976.18,24,16.86,48.0,Male,...,0.750,452.0,1.0,46.3,0,0.0,Pending,1,58,0
1,LN0000002,CUS006128,2023-04-01 18:28:17,Logbook Loan,Debt Consolidation,48612.33,48,14.69,18.0,Female,...,0.610,340.0,2.0,69.5,0,1.0,Approved,0,0,0
2,LN0000003,CUS012097,2024-02-10 21:21:49,Emergency Loan,Medical Emergency,16029.18,60,22.68,44.0,NaN,...,0.651,388.0,NaN,31.0,1,1.0,Rejected,1,201,0
3,LN0000004,CUS012445,2019-09-23 12:44:18,Emergency Loan,Business Expansion,26617.25,12,20.66,36.0,NaN,...,0.080,364.0,0.0,32.1,4,1.0,Pending,1,356,0
4,LN0000005,CUS014263,2020-07-05 03:47:45,SME Loan,Medical Emergency,46958.30,60,29.61,48.0,NaN,...,0.764,353.0,3.0,NaN,12,1.0,Rejected,1,99,0


In [138]:
loan_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   application_id          50000 non-null  object 
 1   customer_id             50000 non-null  object 
 2   application_date        50000 non-null  object 
 3   loan_product            50000 non-null  object 
 4   loan_purpose            50000 non-null  object 
 5   loan_amount_KES         50000 non-null  float64
 6   tenure_months           50000 non-null  int64  
 7   interest_rate_pct       50000 non-null  float64
 8   age                     47557 non-null  float64
 9   gender                  33258 non-null  object 
 10  employment_type         41665 non-null  object 
 11  monthly_income_KES      46960 non-null  float64
 12  debt_to_income_ratio    46960 non-null  float64
 13  credit_score            44089 non-null  float64
 14  num_existing_loans      48050 non-null

In [139]:
loan_copy.isnull().sum()

application_id                0
customer_id                   0
application_date              0
loan_product                  0
loan_purpose                  0
loan_amount_KES               0
tenure_months                 0
interest_rate_pct             0
age                        2443
gender                    16742
employment_type            8335
monthly_income_KES         3040
debt_to_income_ratio       3040
credit_score               5911
num_existing_loans         1950
credit_utilization_pct     5049
num_late_payments_12m         0
has_collateral            16614
application_status            0
defaulted                     0
days_past_due                 0
duplicate_applicant           0
dtype: int64

In [140]:
loan_copy['age'].min()

np.float64(18.0)

**Dropping the Gender column since the credit scoring system is unbiased on gender**

In [141]:
loan_copy=loan_copy.drop(columns=['gender'])

In [142]:
missing_ages=loan_copy[loan_copy['age'].isna()]

**Inputing NaN age with median age within the same loan_product category..**

In [143]:
median_age_loan_products=loan_copy.groupby('loan_product').agg({'age': 'median'})
median_age_loan_products

,age
loan_product,
Emergency Loan,45.0
Group Loan,44.0
Logbook Loan,44.0
Mortgage,44.0
Personal Loan,44.0
SME Loan,44.0


In [144]:
loan_copy['age']=loan_copy['age'].fillna( loan_copy['loan_product'].map(median_age_loan_products['age'] ) )

**Some customers are having contraditing ages with time of loan! Resolving the issue**

In [145]:
loan_copy[loan_copy['customer_id'].duplicated(keep='first')].sort_values(by='customer_id').head()

,application_id,customer_id,application_date,loan_product,loan_purpose,loan_amount_KES,tenure_months,interest_rate_pct,age,employment_type,...,debt_to_income_ratio,credit_score,num_existing_loans,credit_utilization_pct,num_late_payments_12m,has_collateral,application_status,defaulted,days_past_due,duplicate_applicant
8478,LN0008479,CUS000002,2019-04-10 05:23:57,Logbook Loan,Vehicle Purchase,56225.82,18,19.35,47.0,NaN,...,0.327,652.0,NaN,92.8,8,1.0,Approved,0,0,0
18355,LN0018356,CUS000002,2019-09-10 20:44:04,Mortgage,Medical Emergency,17722.69,36,22.02,51.0,Casual Worker,...,0.101,445.0,4.0,85.7,3,1.0,Pending,1,149,0
20755,LN0020756,CUS000003,2020-02-03 19:08:28,SME Loan,Vehicle Purchase,49174.60,24,25.75,26.0,NaN,...,0.612,359.0,5.0,85.9,9,1.0,Approved,0,0,0
31991,LN0031992,CUS000005,2022-12-28 23:23:31,Logbook Loan,Home Improvement,161315.54,6,15.88,63.0,NaN,...,0.076,678.0,3.0,49.9,1,0.0,Rejected,0,0,0
35728,LN0035729,CUS000006,2019-02-15 09:55:46,Group Loan,Business Expansion,70570.71,12,8.02,69.0,Casual Worker,...,0.179,846.0,4.0,92.4,7,NaN,Approved,0,0,0


In [146]:
loan_copy['age'].min(), loan_copy['age'].max()

(np.float64(18.0), np.float64(70.0))

In [147]:
loan_copy['application_date'].max()

'2024-06-29 22:48:41'

In [148]:
unique_customers=pd.DataFrame(loan_copy['customer_id'].unique(),columns=['customer_id'])

In [149]:
len(unique_customers)

14477

In [150]:
unique_customers['age']=pd.Series(np.random.randint(18,71,size=(len(unique_customers) ) ))

In [151]:
unique_customers['age'].min(),unique_customers['age'].max()

(np.int64(18), np.int64(70))

In [152]:
unique_customers.sample()

,customer_id,age
9743,CUS011840,37


In [154]:
unique_customers.sample()

,customer_id,age,max_year
6098,CUS008879,57,2024


In [156]:
unique_customers.sample()

,customer_id,age,max_year
4942,CUS010890,63,2024


In [157]:
unique_customers=unique_customers.set_index('customer_id')

**Mapping the birth year for unique customers to the `loan_copy` data**

In [158]:
loan_copy['age']=loan_copy['customer_id'].map(unique_customers['age'])

**Recomputing the ages of the customers**

In [159]:
loan_copy['application_date']=pd.to_datetime(loan_copy['application_date'])
loan_copy['application_year']=loan_copy['application_date'].dt.year
loan_copy['application_year']=loan_copy['application_year'].astype(int)

In [161]:
loan_copy['age'].min()

np.int64(18)

In [162]:
loan_copy[loan_copy['customer_id']=='CUS001295']

,application_id,customer_id,application_date,loan_product,loan_purpose,loan_amount_KES,tenure_months,interest_rate_pct,age,employment_type,...,credit_score,num_existing_loans,credit_utilization_pct,num_late_payments_12m,has_collateral,application_status,defaulted,days_past_due,duplicate_applicant,application_year
419,LN0000420,CUS001295,2021-02-21 05:55:10,Group Loan,Business Expansion,35036.33,36,26.19,38,Business Owner,...,777.0,1.0,91.4,0,1.0,Approved,0,0,0,2021


**Age column data integrity issue resolved! Proceeding to the next issue:**

In [163]:
loan_copy.isnull().sum()

application_id                0
customer_id                   0
application_date              0
loan_product                  0
loan_purpose                  0
loan_amount_KES               0
tenure_months                 0
interest_rate_pct             0
age                           0
employment_type            8335
monthly_income_KES         3040
debt_to_income_ratio       3040
credit_score               5911
num_existing_loans         1950
credit_utilization_pct     5049
num_late_payments_12m         0
has_collateral            16614
application_status            0
defaulted                     0
days_past_due                 0
duplicate_applicant           0
application_year              0
dtype: int64

**Since the employment-type of the NaN is 100% not known imputing in with `Unknown`**

In [164]:
loan_copy['employment_type']=loan_copy['employment_type'].fillna('Unknown')

In [165]:
loan_copy['employment_type'].value_counts()

employment_type
Self-employed     8454
Salaried          8411
Casual Worker     8335
Unknown           8335
Unemployed        8269
Business Owner    8196
Name: count, dtype: int64

In [166]:
loan_copy.isnull().sum()

application_id                0
customer_id                   0
application_date              0
loan_product                  0
loan_purpose                  0
loan_amount_KES               0
tenure_months                 0
interest_rate_pct             0
age                           0
employment_type               0
monthly_income_KES         3040
debt_to_income_ratio       3040
credit_score               5911
num_existing_loans         1950
credit_utilization_pct     5049
num_late_payments_12m         0
has_collateral            16614
application_status            0
defaulted                     0
days_past_due                 0
duplicate_applicant           0
application_year              0
dtype: int64

In [167]:
loan_copy['num_existing_loans'].min(),loan_copy['num_existing_loans'].median(),loan_copy['num_existing_loans'].max()

(np.float64(0.0), np.float64(3.0), np.float64(5.0))

**Filling in the median value of `num_existing_loans` `column` for `NaNs`**

In [168]:
loan_copy['num_existing_loans']=loan_copy['num_existing_loans'].fillna( loan_copy['num_existing_loans'].median() )

In [169]:
loan_copy.isnull().sum()

application_id                0
customer_id                   0
application_date              0
loan_product                  0
loan_purpose                  0
loan_amount_KES               0
tenure_months                 0
interest_rate_pct             0
age                           0
employment_type               0
monthly_income_KES         3040
debt_to_income_ratio       3040
credit_score               5911
num_existing_loans            0
credit_utilization_pct     5049
num_late_payments_12m         0
has_collateral            16614
application_status            0
defaulted                     0
days_past_due                 0
duplicate_applicant           0
application_year              0
dtype: int64

**Treating blanks in `has_collateral` as `Unknown`**

In [170]:
loan_copy['has_collateral']=loan_copy['has_collateral'].fillna('Unknown')

In [171]:
loan_copy.isnull().sum()

application_id               0
customer_id                  0
application_date             0
loan_product                 0
loan_purpose                 0
loan_amount_KES              0
tenure_months                0
interest_rate_pct            0
age                          0
employment_type              0
monthly_income_KES        3040
debt_to_income_ratio      3040
credit_score              5911
num_existing_loans           0
credit_utilization_pct    5049
num_late_payments_12m        0
has_collateral               0
application_status           0
defaulted                    0
days_past_due                0
duplicate_applicant          0
application_year             0
dtype: int64

**Remaining blank values will be recomputed later...Proceeding to the `transactions_history data`**

In [172]:
transaction_copy.head()

,txn_id,customer_id,txn_date,txn_type,amount_KES,direction,balance_after_KES,bounce_flag,bank
0,TH00000001,CUS001171,2024-01-17 22:25:58,Salary Credit,11244.09,Debit,74761.54,0,Absa
1,TH00000002,CUS010054,2022-11-17 18:01:47,Transfer,24436.71,Credit,210668.52,0,Absa
2,TH00000003,CUS005097,2018-07-09 10:38:04,Transfer,17435.40,Debit,365592.14,0,Equity Bank
3,TH00000004,CUS010864,2021-08-04 12:17:12,Loan Repayment,1273.76,Credit,60118.40,0,Stanbic
4,TH00000005,CUS004175,2022-09-01 13:52:56,Mobile Money,7594.04,Credit,NaN,0,DTB


In [173]:
transaction_copy.isnull().sum()

txn_id                  0
customer_id             0
txn_date                0
txn_type                0
amount_KES              0
direction               0
balance_after_KES    4669
bounce_flag             0
bank                    0
dtype: int64

**Since the blanks of `balance_after_KES` are known; they will excluded...**

In [174]:
transaction_copy=transaction_copy.dropna()

In [175]:
transaction_copy.isnull().sum()

txn_id               0
customer_id          0
txn_date             0
txn_type             0
amount_KES           0
direction            0
balance_after_KES    0
bounce_flag          0
bank                 0
dtype: int64

In [176]:
transaction_copy['txn_date']=pd.to_datetime(transaction_copy['txn_date'])

**Calculating customers monthly debt obligations**

In [177]:
transaction_copy.head()

,txn_id,customer_id,txn_date,txn_type,amount_KES,direction,balance_after_KES,bounce_flag,bank
0,TH00000001,CUS001171,2024-01-17 22:25:58,Salary Credit,11244.09,Debit,74761.54,0,Absa
1,TH00000002,CUS010054,2022-11-17 18:01:47,Transfer,24436.71,Credit,210668.52,0,Absa
2,TH00000003,CUS005097,2018-07-09 10:38:04,Transfer,17435.40,Debit,365592.14,0,Equity Bank
3,TH00000004,CUS010864,2021-08-04 12:17:12,Loan Repayment,1273.76,Credit,60118.40,0,Stanbic
5,TH00000006,CUS013440,2020-01-19 19:55:15,Utility,4002.65,Credit,241076.95,0,NCBA


In [178]:
len(transaction_copy)

75331

In [179]:
len(transaction_copy[ (transaction_copy['txn_type']=='Loan Repayment') & (transaction_copy['direction']=='Debit') ] )

5291

In [180]:
loan_repayment_data=transaction_copy[transaction_copy['txn_type']=='Loan Repayment']
loan_repayment_debit_data=loan_repayment_data[loan_repayment_data['direction']=='Debit']

In [181]:
loan_repayment_debit_data.head()

,txn_id,customer_id,txn_date,txn_type,amount_KES,direction,balance_after_KES,bounce_flag,bank
19,TH00000020,CUS012018,2019-09-13 12:17:40,Loan Repayment,765.54,Debit,44893.95,0,Stanbic
36,TH00000037,CUS003137,2019-05-26 17:13:45,Loan Repayment,12507.31,Debit,143907.24,0,Absa
41,TH00000042,CUS011507,2019-01-07 22:47:30,Loan Repayment,44421.02,Debit,264625.50,0,Family Bank
84,TH00000085,CUS014037,2023-02-23 06:47:17,Loan Repayment,11150.72,Debit,412568.58,0,DTB
87,TH00000088,CUS004989,2021-07-30 22:54:55,Loan Repayment,13249.45,Debit,236265.69,0,Equity Bank


**Aggregating per customers to get monthly debit for Loan Repayment transactions**

In [182]:
loan_repayment_debit_data=loan_repayment_debit_data.groupby('customer_id').agg( {'amount_KES': 'sum', 'txn_id': 'count'} )
loan_repayment_debit_data.columns=['total_debit_loan_repayment','total_transactions_made']

In [183]:
loan_repayment_debit_data['monthly_debit_amount']=loan_repayment_debit_data['total_debit_loan_repayment'] / loan_repayment_debit_data['total_transactions_made']

In [184]:
loan_repayment_debit_data

,total_debit_loan_repayment,total_transactions_made,monthly_debit_amount
customer_id,,,
CUS000002,29694.76,1,29694.760
CUS000006,28958.37,2,14479.185
CUS000009,3367.15,1,3367.150
CUS000011,4434.36,1,4434.360
CUS000012,18039.75,1,18039.750
...,...,...,...
CUS014986,5656.49,1,5656.490
CUS014994,56978.50,2,28489.250
CUS014995,2157.63,1,2157.630


**Mapping the `monthly_debit_amount` from loan_repayment_debit_data to transaction_copy**

In [185]:
transaction_copy['monthly_debit_amount']=transaction_copy['customer_id'].map(loan_repayment_debit_data['monthly_debit_amount'] )

In [186]:
transaction_copy[transaction_copy['customer_id']=='CUS015000']

,txn_id,customer_id,txn_date,txn_type,amount_KES,direction,balance_after_KES,bounce_flag,bank,monthly_debit_amount
2735,TH00002736,CUS015000,2019-11-04 06:52:06,ATM,3406.07,Credit,361273.85,0,NCBA,9435.05
10821,TH00010822,CUS015000,2020-11-10 08:05:39,Utility,6802.55,Credit,92989.05,0,Coop Bank,9435.05
14732,TH00014733,CUS015000,2022-07-23 09:51:30,Loan Repayment,9435.05,Debit,154394.31,1,Family Bank,9435.05
79356,TH00079357,CUS015000,2018-11-21 05:50:34,Mobile Money,29611.06,Credit,330268.82,0,Absa,9435.05


In [187]:
transaction_copy['monthly_debit_amount']=transaction_copy['monthly_debit_amount'].fillna(0)

In [188]:
transaction_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75331 entries, 0 to 79999
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   txn_id                75331 non-null  object        
 1   customer_id           75331 non-null  object        
 2   txn_date              75331 non-null  datetime64[ns]
 3   txn_type              75331 non-null  object        
 4   amount_KES            75331 non-null  float64       
 5   direction             75331 non-null  object        
 6   balance_after_KES     75331 non-null  float64       
 7   bounce_flag           75331 non-null  int64         
 8   bank                  75331 non-null  object        
 9   monthly_debit_amount  75331 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(5)
memory usage: 6.3+ MB


**Mapping the `monthly_debit_amount` from transaction_copy to loan_data**

In [189]:
loan_rep_cus_data=transaction_copy.groupby('customer_id').agg({'monthly_debit_amount': 'mean'})
loan_rep_cus_data

,monthly_debit_amount
customer_id,
CUS000001,0.000
CUS000002,29694.760
CUS000003,0.000
CUS000006,14479.185
CUS000007,0.000
...,...
CUS014996,0.000
CUS014997,0.000
CUS014998,9531.990


In [190]:
loan_copy['monthly_debit_amount']=loan_copy['customer_id'].map(loan_rep_cus_data['monthly_debit_amount'] )

In [191]:
loan_copy.isnull().sum()

application_id               0
customer_id                  0
application_date             0
loan_product                 0
loan_purpose                 0
loan_amount_KES              0
tenure_months                0
interest_rate_pct            0
age                          0
employment_type              0
monthly_income_KES        3040
debt_to_income_ratio      3040
credit_score              5911
num_existing_loans           0
credit_utilization_pct    5049
num_late_payments_12m        0
has_collateral               0
application_status           0
defaulted                    0
days_past_due                0
duplicate_applicant          0
application_year             0
monthly_debit_amount       869
dtype: int64

**Estimating customer's monthly income for those blank rows with `[Sum(Salary Credit =='Credit') / total_transactions] per customer_transactions`**

In [192]:
transaction_copy.head()

,txn_id,customer_id,txn_date,txn_type,amount_KES,direction,balance_after_KES,bounce_flag,bank,monthly_debit_amount
0,TH00000001,CUS001171,2024-01-17 22:25:58,Salary Credit,11244.09,Debit,74761.54,0,Absa,0.00
1,TH00000002,CUS010054,2022-11-17 18:01:47,Transfer,24436.71,Credit,210668.52,0,Absa,0.00
2,TH00000003,CUS005097,2018-07-09 10:38:04,Transfer,17435.40,Debit,365592.14,0,Equity Bank,0.00
3,TH00000004,CUS010864,2021-08-04 12:17:12,Loan Repayment,1273.76,Credit,60118.40,0,Stanbic,2352.12
5,TH00000006,CUS013440,2020-01-19 19:55:15,Utility,4002.65,Credit,241076.95,0,NCBA,22545.07


In [193]:
salary_credit_data=transaction_copy[transaction_copy['txn_type']=='Salary Credit']
creditted_salary_data=salary_credit_data[salary_credit_data['direction']=='Credit']

In [194]:
creditted_salary_data.head()

,txn_id,customer_id,txn_date,txn_type,amount_KES,direction,balance_after_KES,bounce_flag,bank,monthly_debit_amount
59,TH00000060,CUS013888,2020-05-24 11:40:53,Salary Credit,22093.35,Credit,244107.74,0,KCB,0.0
60,TH00000061,CUS012859,2019-04-01 10:55:14,Salary Credit,24771.06,Credit,31709.50,0,Equity Bank,0.0
97,TH00000098,CUS010049,2020-12-20 08:15:42,Salary Credit,2635.05,Credit,362855.67,0,Equity Bank,0.0
108,TH00000109,CUS011669,2019-07-08 04:09:47,Salary Credit,19222.96,Credit,415615.11,0,NCBA,0.0
113,TH00000114,CUS007136,2023-05-23 15:50:40,Salary Credit,12524.65,Credit,455556.65,0,KCB,0.0


In [195]:
creditted_salary_data=creditted_salary_data.groupby('customer_id').agg( {'amount_KES': 'sum','txn_id': 'count'} )

In [196]:
creditted_salary_data['estimate_monthly_income']=creditted_salary_data['amount_KES'] / creditted_salary_data['txn_id']

In [197]:
creditted_salary_data

,amount_KES,txn_id,estimate_monthly_income
customer_id,,,
CUS000002,4788.90,1,4788.90
CUS000003,15637.16,1,15637.16
CUS000009,7924.53,1,7924.53
CUS000010,1957.27,1,1957.27
CUS000018,6605.21,1,6605.21
...,...,...,...
CUS014983,12885.55,1,12885.55
CUS014984,33681.54,1,33681.54
CUS014986,3435.03,1,3435.03


**Mapping the estimate_monthly income to the loan_copy data...**

In [198]:
loan_copy['estimate_monthly_income']=loan_copy['customer_id'].map(creditted_salary_data['estimate_monthly_income'] )

In [199]:
loan_copy.isnull().sum()

application_id                 0
customer_id                    0
application_date               0
loan_product                   0
loan_purpose                   0
loan_amount_KES                0
tenure_months                  0
interest_rate_pct              0
age                            0
employment_type                0
monthly_income_KES          3040
debt_to_income_ratio        3040
credit_score                5911
num_existing_loans             0
credit_utilization_pct      5049
num_late_payments_12m          0
has_collateral                 0
application_status             0
defaulted                      0
days_past_due                  0
duplicate_applicant            0
application_year               0
monthly_debit_amount         869
estimate_monthly_income    31934
dtype: int64

In [200]:
loan_copy['monthly_income_KES']=np.where(loan_copy['monthly_income_KES'].isnull(),loan_copy['estimate_monthly_income'], loan_copy['monthly_income_KES'])

In [201]:
loan_copy.isnull().sum()

application_id                 0
customer_id                    0
application_date               0
loan_product                   0
loan_purpose                   0
loan_amount_KES                0
tenure_months                  0
interest_rate_pct              0
age                            0
employment_type                0
monthly_income_KES          1924
debt_to_income_ratio        3040
credit_score                5911
num_existing_loans             0
credit_utilization_pct      5049
num_late_payments_12m          0
has_collateral                 0
application_status             0
defaulted                      0
days_past_due                  0
duplicate_applicant            0
application_year               0
monthly_debit_amount         869
estimate_monthly_income    31934
dtype: int64

**Since there are 1924 records with missing values for `monthly_income_KES`..They will be dropped....**

In [202]:
loan_copy=loan_copy.dropna(subset=['monthly_income_KES'])

In [203]:
loan_copy.isnull().sum()

application_id                 0
customer_id                    0
application_date               0
loan_product                   0
loan_purpose                   0
loan_amount_KES                0
tenure_months                  0
interest_rate_pct              0
age                            0
employment_type                0
monthly_income_KES             0
debt_to_income_ratio        1116
credit_score                5692
num_existing_loans             0
credit_utilization_pct      4849
num_late_payments_12m          0
has_collateral                 0
application_status             0
defaulted                      0
days_past_due                  0
duplicate_applicant            0
application_year               0
monthly_debit_amount         817
estimate_monthly_income    30010
dtype: int64

**Recomputing `debt_to_income_ratio: (monthly_debit_amount/ monthly_income)`**

In [204]:
loan_copy['monthly_debit_amount']=loan_copy['monthly_debit_amount'].fillna(0)

In [205]:
loan_copy['debt_to_income_ratio']=loan_copy['monthly_debit_amount'] / loan_copy['monthly_income_KES']

In [206]:
loan_copy.isnull().sum()

application_id                 0
customer_id                    0
application_date               0
loan_product                   0
loan_purpose                   0
loan_amount_KES                0
tenure_months                  0
interest_rate_pct              0
age                            0
employment_type                0
monthly_income_KES             0
debt_to_income_ratio           0
credit_score                5692
num_existing_loans             0
credit_utilization_pct      4849
num_late_payments_12m          0
has_collateral                 0
application_status             0
defaulted                      0
days_past_due                  0
duplicate_applicant            0
application_year               0
monthly_debit_amount           0
estimate_monthly_income    30010
dtype: int64

In [207]:
loan_copy['num_late_payments_12m'].value_counts()

num_late_payments_12m
3     3799
8     3786
0     3770
7     3769
1     3752
10    3751
6     3707
11    3690
4     3679
12    3628
9     3606
2     3581
5     3558
Name: count, dtype: int64

**Redo-ing the credit-score data for it to be logical `-- if late payments < 3 --score(700-850); late payments >=3 and < 6 --score(600-650); late payments >=6 and < 8 --score(400-500);  else score(300-400)`**

In [208]:
scores=[]
late_payments=loan_copy['num_late_payments_12m']

In [209]:
for i in late_payments:
    if i < 3:
        j=np.random.randint(700,851)
        scores.append(j)
    elif (i >= 3) and (i < 6):
        j=np.random.randint(500,650)
        scores.append(j)
    elif (i >= 6) and (i < 8):
        j=np.random.randint(400,500)
        scores.append(j)
    else:
        j=np.random.randint(300,400)
        scores.append(j)

In [210]:
len(scores), len(loan_copy)

(48076, 48076)

In [211]:
loan_copy['credit_score']=scores

In [212]:
loan_copy.isnull().sum()

application_id                 0
customer_id                    0
application_date               0
loan_product                   0
loan_purpose                   0
loan_amount_KES                0
tenure_months                  0
interest_rate_pct              0
age                            0
employment_type                0
monthly_income_KES             0
debt_to_income_ratio           0
credit_score                   0
num_existing_loans             0
credit_utilization_pct      4849
num_late_payments_12m          0
has_collateral                 0
application_status             0
defaulted                      0
days_past_due                  0
duplicate_applicant            0
application_year               0
monthly_debit_amount           0
estimate_monthly_income    30010
dtype: int64

**Creating a `Credit_score band column` with logic: `score(700-850)--'low_risk'; score(550-700) -- 'medium_risk'; else --'high_risk'....`..**

In [213]:
credit_score_bands=[]
credit_scores=loan_copy['credit_score']

In [214]:
for i in credit_scores:
    if i > 700:
        j='low_risk'
        credit_score_bands.append(j)
    elif (i >= 550) and (i < 700):
        j='medium_risk'
        credit_score_bands.append(j)
    else:
        j='high_risk'
        credit_score_bands.append(j)

In [215]:
loan_copy['credit_score_bands']=credit_score_bands

In [216]:
loan_copy.isnull().sum()

application_id                 0
customer_id                    0
application_date               0
loan_product                   0
loan_purpose                   0
loan_amount_KES                0
tenure_months                  0
interest_rate_pct              0
age                            0
employment_type                0
monthly_income_KES             0
debt_to_income_ratio           0
credit_score                   0
num_existing_loans             0
credit_utilization_pct      4849
num_late_payments_12m          0
has_collateral                 0
application_status             0
defaulted                      0
days_past_due                  0
duplicate_applicant            0
application_year               0
monthly_debit_amount           0
estimate_monthly_income    30010
credit_score_bands             0
dtype: int64

**Getting `credit_score_bands` `[credit_utilization_pct] median value`...**

In [217]:
median_bands_scores=loan_copy.groupby('credit_score_bands').agg({'credit_utilization_pct': 'median'} )
median_bands_scores

,credit_utilization_pct
credit_score_bands,
high_risk,50.7
low_risk,50.5
medium_risk,50.5


**Filling Missing values in the `credit_utilization_pct` column with `Imputing using median within credit_score band`**

In [218]:
loan_copy['credit_utilization_pct']=loan_copy['credit_utilization_pct'].fillna( loan_copy['credit_score_bands'].map(median_bands_scores['credit_utilization_pct']) )

In [219]:
loan_copy.isnull().sum()

application_id                 0
customer_id                    0
application_date               0
loan_product                   0
loan_purpose                   0
loan_amount_KES                0
tenure_months                  0
interest_rate_pct              0
age                            0
employment_type                0
monthly_income_KES             0
debt_to_income_ratio           0
credit_score                   0
num_existing_loans             0
credit_utilization_pct         0
num_late_payments_12m          0
has_collateral                 0
application_status             0
defaulted                      0
days_past_due                  0
duplicate_applicant            0
application_year               0
monthly_debit_amount           0
estimate_monthly_income    30010
credit_score_bands             0
dtype: int64

In [220]:
#dropping the `estimate_monthly_income`..
loan_copy=loan_copy.drop(columns=['estimate_monthly_income'])

In [221]:
loan_copy.isnull().sum()

application_id            0
customer_id               0
application_date          0
loan_product              0
loan_purpose              0
loan_amount_KES           0
tenure_months             0
interest_rate_pct         0
age                       0
employment_type           0
monthly_income_KES        0
debt_to_income_ratio      0
credit_score              0
num_existing_loans        0
credit_utilization_pct    0
num_late_payments_12m     0
has_collateral            0
application_status        0
defaulted                 0
days_past_due             0
duplicate_applicant       0
application_year          0
monthly_debit_amount      0
credit_score_bands        0
dtype: int64

In [222]:
loan_copy[loan_copy['defaulted']== 1]

,application_id,customer_id,application_date,loan_product,loan_purpose,loan_amount_KES,tenure_months,interest_rate_pct,age,employment_type,...,credit_utilization_pct,num_late_payments_12m,has_collateral,application_status,defaulted,days_past_due,duplicate_applicant,application_year,monthly_debit_amount,credit_score_bands
0,LN0000001,CUS002206,2020-04-07 16:07:23,Mortgage,Business Expansion,26976.18,24,16.86,40,Business Owner,...,46.3,0,0.0,Pending,1,58,0,2020,15841.71,low_risk
2,LN0000003,CUS012097,2024-02-10 21:21:49,Emergency Loan,Medical Emergency,16029.18,60,22.68,30,Self-employed,...,31.0,1,1.0,Rejected,1,201,0,2024,0.00,low_risk
3,LN0000004,CUS012445,2019-09-23 12:44:18,Emergency Loan,Business Expansion,26617.25,12,20.66,31,Unknown,...,32.1,4,1.0,Pending,1,356,0,2019,0.00,medium_risk
4,LN0000005,CUS014263,2020-07-05 03:47:45,SME Loan,Medical Emergency,46958.30,60,29.61,46,Salaried,...,50.7,12,1.0,Rejected,1,99,0,2020,0.00,high_risk
5,LN0000006,CUS009300,2021-10-20 23:07:42,Group Loan,Business Expansion,53244.28,36,25.32,42,Self-employed,...,91.0,3,Unknown,Pending,1,320,0,2021,15323.51,medium_risk
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49990,LN0049991,CUS001965,2022-05-25 01:11:43,Mortgage,Agriculture,118017.51,24,18.71,53,Casual Worker,...,96.6,3,Unknown,Approved,1,204,0,2022,0.00,high_risk
49992,LN0049993,CUS001552,2020-04-06 03:07:24,Emergency Loan,Personal,35293.40,18,21.32,55,Unknown,...,50.7,7,0.0,Approved,1,241,0,2020,0.00,high_risk
49993,LN0049994,CUS003815,2020-05-14 14:47:41,Emergency Loan,Debt Consolidation,34417.44,12,11.32,56,Unknown,...,24.1,1,Unknown,Approved,1,2,1,2020,0.00,low_risk
49995,LN0049996,CUS010750,2022-06-17 01:58:16,Emergency Loan,Home Improvement,92368.45,48,22.56,44,Unemployed,...,64.2,3,1.0,Pending,1,173,0,2022,10188.77,medium_risk


In [225]:
loan_copy['tenure_months'].min(),loan_copy['tenure_months'].median(),loan_copy['tenure_months'].max(),loan_copy['tenure_months'].mean()

(np.int64(6), np.float64(24.0), np.int64(60), np.float64(29.253806473084282))

**Calculating the tenure scores....**

In [234]:
tenures=loan_copy['tenure_months']
score_tenures=[]

for i in tenures:
    if i < 12:
        j=np.random.randint(0,12)
        score_tenures.append(j)
    elif (i >= 12) and (i < 18):
        j=np.random.randint(12,18)
        score_tenures.append(j)
    elif (i >= 18) and (i < 24):
        j=np.random.randint(18,24)
        score_tenures.append(j)
    else:
        j=np.random.randint(25,60)
        score_tenures.append(j)

In [247]:
loan_copy['scores_tenure']=score_tenures
loan_copy['scores_tenure']=( (loan_copy['scores_tenure'] / 2) / 100)

**Calculating DTI scores...**

In [243]:
loan_copy['debt_to_income_ratio'].min(),loan_copy['debt_to_income_ratio'].median(), loan_copy['debt_to_income_ratio'].max(),loan_copy['debt_to_income_ratio'].mean()

(np.float64(0.0),
 np.float64(0.0),
 np.float64(86.11741270213265),
 np.float64(0.3183434870966965))

In [244]:
dti=loan_copy['debt_to_income_ratio']
score_dti=[]

for i in dti:
    if i < 0.30:
        j=np.random.randint(0,30)
        score_dti.append(j)
    elif (i >= 0.30) and (i < 0.42):
        j=np.random.randint(30,42)
        score_dti.append(j)
    else:
        j=np.random.randint(43,60)
        score_dti.append(j)

In [248]:
loan_copy['score_dti']=score_dti
loan_copy['score_dti']=( (loan_copy['score_dti'] / 2) / 100)

**Calculating existing_loans scores...**

In [255]:
loan_copy['num_existing_loans'].min(),loan_copy['num_existing_loans'].median(),loan_copy['num_existing_loans'].max(),loan_copy['num_existing_loans'].mean()

(np.float64(0.0),
 np.float64(3.0),
 np.float64(5.0),
 np.float64(2.5266869123887177))

In [256]:
existing_loans=loan_copy['num_existing_loans']
score_existing_loans=[]

for i in existing_loans:
    if i < 3:
        j=np.random.randint(0,30)
        score_existing_loans.append(j)
    elif (i >= 3) and (i < 5):
        j=np.random.randint(30,50)
        score_existing_loans.append(j)
    else:
        j=np.random.randint(50,70)
        score_existing_loans.append(j)

In [257]:
loan_copy['score_existing_loans']=score_existing_loans
loan_copy['score_existing_loans']=( (loan_copy['score_existing_loans'] / 2) / 100)

**Calculating credit_utilization_pct scores...**

In [259]:
loan_copy['credit_utilization_pct'].min(),loan_copy['credit_utilization_pct'].median(),loan_copy['credit_utilization_pct'].max(),loan_copy['credit_utilization_pct'].mean()

(np.float64(0.0),
 np.float64(50.7),
 np.float64(100.0),
 np.float64(50.36619311090773))

In [260]:
credit_utilization_pct=loan_copy['credit_utilization_pct']
score_credit_utilization_pct=[]

for i in credit_utilization_pct:
    if i < 30:
        j=np.random.randint(0,30)
        score_credit_utilization_pct.append(j)
    elif (i >= 30) and (i < 60):
        j=np.random.randint(30,50)
        score_credit_utilization_pct.append(j)
    elif (i >= 60) and (i < 90):
        j=np.random.randint(60,90)
        score_credit_utilization_pct.append(j)
    else:
        j=np.random.randint(90,100)
        score_credit_utilization_pct.append(j)

In [261]:
loan_copy['score_credit_utilization_pct']=score_credit_utilization_pct
loan_copy['score_credit_utilization_pct']=( (loan_copy['score_credit_utilization_pct'] / 2) / 100)

**Calculating Late payments scores...**

In [262]:
loan_copy['num_late_payments_12m'].min(),loan_copy['num_late_payments_12m'].median(), loan_copy['num_late_payments_12m'].max(),loan_copy['num_late_payments_12m'].mean()

(np.int64(0), np.float64(6.0), np.int64(12), np.float64(5.986770946002164))

In [263]:
late_pays=loan_copy['num_late_payments_12m']
score_late_pays=[]

for i in late_pays:
    if i < 3:
        j=np.random.randint(0,30)
        score_late_pays.append(j)
    elif (i >= 3) and (i < 6):
        j=np.random.randint(30,60)
        score_late_pays.append(j)
    elif (i >= 6) and (i < 9):
        j=np.random.randint(60,90)
        score_late_pays.append(j)
    else:
        j=np.random.randint(90,100)
        score_late_pays.append(j)

In [264]:
loan_copy['score_late_pays']=score_late_pays
loan_copy['score_late_pays']=( (loan_copy['score_late_pays'] / 2) / 100)

**Calculating credit_score_bands_scores...**

In [267]:
loan_copy['credit_score_bands'].value_counts()

credit_score_bands
high_risk      29729
low_risk       11018
medium_risk     7329
Name: count, dtype: int64

In [268]:
credit_score_bands=loan_copy['credit_score_bands']
score_credit_score_bands=[]

for i in credit_score_bands:
    if i == 'low_risk':
        j=np.random.randint(0,30)
        score_credit_score_bands.append(j)
    elif i=='medium_risk':
        j=np.random.randint(30,60)
        score_credit_score_bands.append(j)
    else:
        j=np.random.randint(80,100)
        score_credit_score_bands.append(j)

In [269]:
loan_copy['score_credit_score_bands']=score_credit_score_bands
loan_copy['score_credit_score_bands']=( (loan_copy['score_credit_score_bands'] / 2) / 100)

**Putting all the scores all together...**

In [280]:
loan_copy['all_scores_included']=loan_copy['scores_tenure'] + loan_copy['score_dti'] + loan_copy['score_late_pays'] + loan_copy['score_existing_loans'] + loan_copy['score_credit_utilization_pct'] + loan_copy['score_credit_score_bands']
loan_copy['all_scores_included']=round(loan_copy['all_scores_included'])

In [281]:
loan_copy['all_scores_included'].value_counts()

all_scores_included
1.0    32258
2.0    14885
0.0      933
Name: count, dtype: int64

In [276]:
loan_copy['all_scores_included'].min(),loan_copy['all_scores_included'].median(), loan_copy['all_scores_included'].max(),loan_copy['all_scores_included'].mean()

(np.float64(0.0),
 np.float64(1.0),
 np.float64(2.0),
 np.float64(1.290207171977702))

**if all_scores_included >= 2; 1 else 0**

In [282]:
loan_copy['all_scores_included']=np.where(loan_copy['all_scores_included'] >=2 ,1 , 0)

In [283]:
loan_copy['all_scores_included'].value_counts()

all_scores_included
0    33191
1    14885
Name: count, dtype: int64

In [286]:
loan_copy.head()

,application_id,customer_id,application_date,loan_product,loan_purpose,loan_amount_KES,tenure_months,interest_rate_pct,age,employment_type,...,application_year,monthly_debit_amount,credit_score_bands,scores_tenure,score_dti,score_late_pays,score_existing_loans,score_credit_utilization_pct,score_credit_score_bands,all_scores_included
0,LN0000001,CUS002206,2020-04-07 16:07:23,Mortgage,Business Expansion,26976.18,24,16.86,40,Business Owner,...,2020,15841.71,low_risk,0.170,0.290,0.135,0.130,0.225,0.060,0
1,LN0000002,CUS006128,2023-04-01 18:28:17,Logbook Loan,Debt Consolidation,48612.33,48,14.69,63,Salaried,...,2023,0.00,low_risk,0.250,0.110,0.000,0.020,0.410,0.000,0
2,LN0000003,CUS012097,2024-02-10 21:21:49,Emergency Loan,Medical Emergency,16029.18,60,22.68,30,Self-employed,...,2024,0.00,low_risk,0.165,0.005,0.015,0.235,0.235,0.040,0
3,LN0000004,CUS012445,2019-09-23 12:44:18,Emergency Loan,Business Expansion,26617.25,12,20.66,31,Unknown,...,2019,0.00,medium_risk,0.070,0.010,0.280,0.055,0.235,0.250,0
4,LN0000005,CUS014263,2020-07-05 03:47:45,SME Loan,Medical Emergency,46958.30,60,29.61,46,Salaried,...,2020,0.00,high_risk,0.155,0.140,0.490,0.155,0.200,0.455,1


In [287]:
loan_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48076 entries, 0 to 49999
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   application_id                48076 non-null  object        
 1   customer_id                   48076 non-null  object        
 2   application_date              48076 non-null  datetime64[ns]
 3   loan_product                  48076 non-null  object        
 4   loan_purpose                  48076 non-null  object        
 5   loan_amount_KES               48076 non-null  float64       
 6   tenure_months                 48076 non-null  int64         
 7   interest_rate_pct             48076 non-null  float64       
 8   age                           48076 non-null  int64         
 9   employment_type               48076 non-null  object        
 10  monthly_income_KES            48076 non-null  float64       
 11  debt_to_income_ratio          480

**Importing the necessary Libraries for modeling..**

In [288]:
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report,recall_score,precision_score,f1_score,accuracy_score,precision_recall_curve,PrecisionRecallDisplay

In [289]:
loan_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48076 entries, 0 to 49999
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   application_id                48076 non-null  object        
 1   customer_id                   48076 non-null  object        
 2   application_date              48076 non-null  datetime64[ns]
 3   loan_product                  48076 non-null  object        
 4   loan_purpose                  48076 non-null  object        
 5   loan_amount_KES               48076 non-null  float64       
 6   tenure_months                 48076 non-null  int64         
 7   interest_rate_pct             48076 non-null  float64       
 8   age                           48076 non-null  int64         
 9   employment_type               48076 non-null  object        
 10  monthly_income_KES            48076 non-null  float64       
 11  debt_to_income_ratio          480

**Creating customers_data from loan_data**

In [562]:
customers_data_agg=loan_copy.groupby('customer_id').agg( {'age':'median','application_id': 'count','loan_amount_KES': 'mean', 'monthly_income_KES': 'mean','num_existing_loans': 'mean','num_late_payments_12m':'mean','monthly_debit_amount': 'mean', 'credit_score': 'median','credit_utilization_pct' : 'mean'} ).reset_index()

In [563]:
# rounding the variables to 1 decimal places...
customers_data_agg['credit_score']= round(customers_data_agg['credit_score'])
customers_data_agg['num_existing_loans']= round(customers_data_agg['num_existing_loans'])
customers_data_agg['num_late_payments_12m']= round(customers_data_agg['num_late_payments_12m'])
customers_data_agg['loan_amount_KES']= round(customers_data_agg['loan_amount_KES'])
customers_data_agg['monthly_income_KES']= round(customers_data_agg['monthly_income_KES'])
customers_data_agg['monthly_debit_amount']= round(customers_data_agg['monthly_debit_amount'])
customers_data_agg['credit_utilization_pct']= round(customers_data_agg['credit_utilization_pct'])

In [564]:
customers_data_agg

,customer_id,age,application_id,loan_amount_KES,monthly_income_KES,num_existing_loans,num_late_payments_12m,monthly_debit_amount,credit_score,credit_utilization_pct
0,CUS000001,35.0,1,73257.0,102774.0,2.0,4.0,0.0,577.0,93.0
1,CUS000002,18.0,3,25878.0,56448.0,3.0,6.0,29695.0,383.0,60.0
2,CUS000003,43.0,2,206404.0,35163.0,5.0,9.0,0.0,376.0,68.0
3,CUS000005,64.0,2,214247.0,27932.0,2.0,6.0,0.0,544.0,36.0
4,CUS000006,60.0,5,130731.0,25822.0,4.0,7.0,14479.0,365.0,50.0
...,...,...,...,...,...,...,...,...,...,...
14367,CUS014995,24.0,4,246294.0,41782.0,3.0,8.0,2158.0,406.0,51.0
14368,CUS014996,49.0,2,206688.0,12370.0,4.0,7.0,0.0,478.0,35.0
14369,CUS014998,60.0,2,68796.0,23415.0,2.0,8.0,9532.0,418.0,44.0
14370,CUS014999,42.0,2,52609.0,19424.0,5.0,10.0,0.0,358.0,35.0


**Calculating each customers default values...**

In [565]:
defaulted_customers_agg=loan_copy.groupby('customer_id')['all_scores_included'].median().reset_index()

defaulted_customers_agg['all_scores_included']=np.where( defaulted_customers_agg['all_scores_included']==0.5,1,defaulted_customers_agg['all_scores_included'])
defaulted_customers_agg=defaulted_customers_agg.set_index('customer_id')

In [566]:
defaulted_customers_agg['all_scores_included'].value_counts()

all_scores_included
0.0    9849
1.0    4523
Name: count, dtype: int64

**Mapping the results of defaulted_values to customers_data**

In [567]:
customers_data_agg['defaulted']=customers_data_agg['customer_id'].map(defaulted_customers_agg['all_scores_included'])

In [568]:
customers_data_agg=customers_data_agg.rename(columns={'monthly_debit_amount':'monthly_loan_repayment_debit_amount'})

In [569]:
customers_data_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14372 entries, 0 to 14371
Data columns (total 11 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   customer_id                          14372 non-null  object 
 1   age                                  14372 non-null  float64
 2   application_id                       14372 non-null  int64  
 3   loan_amount_KES                      14372 non-null  float64
 4   monthly_income_KES                   14372 non-null  float64
 5   num_existing_loans                   14372 non-null  float64
 6   num_late_payments_12m                14372 non-null  float64
 7   monthly_loan_repayment_debit_amount  14372 non-null  float64
 8   credit_score                         14372 non-null  float64
 9   credit_utilization_pct               14372 non-null  float64
 10  defaulted                            14372 non-null  float64
dtypes: float64(9), int64(1), obj

**Customer's `Annual-income= monthly_income * 12 months`**

In [570]:
customers_data_agg['annual_income_KES']=(customers_data_agg['monthly_income_KES'] * 12)

**DTI = total monthly debt obligations ÷ monthly income**

In [571]:
customers_data_agg['debt_to_income_ratio%']=( (customers_data_agg['monthly_loan_repayment_debit_amount'] / customers_data_agg['monthly_income_KES']) *100 )

**LTI= loan_amount ÷ annual_income**

In [572]:
customers_data_agg['loan_to_income_ratio%']=( (customers_data_agg['loan_amount_KES'] / customers_data_agg['annual_income_KES']) * 100 )

**Adding the following column--`employment_type` to customers_data...**

In [573]:
loan_copy=loan_copy.sort_values(by='application_date')

In [574]:
employment_type_customers=loan_copy.groupby('customer_id')['employment_type'].first().reset_index()
employment_type_customers=employment_type_customers.set_index('customer_id')

In [575]:
employment_type_customers['employment_type'].value_counts()

employment_type
Self-employed     2537
Salaried          2393
Casual Worker     2379
Business Owner    2368
Unemployed        2355
Unknown           2340
Name: count, dtype: int64

**Mapping the results of employment_type_customers to customers_data**

In [576]:
customers_data_agg['employment_type']=customers_data_agg['customer_id'].map(employment_type_customers['employment_type'])

In [577]:
customers_data_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14372 entries, 0 to 14371
Data columns (total 15 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   customer_id                          14372 non-null  object 
 1   age                                  14372 non-null  float64
 2   application_id                       14372 non-null  int64  
 3   loan_amount_KES                      14372 non-null  float64
 4   monthly_income_KES                   14372 non-null  float64
 5   num_existing_loans                   14372 non-null  float64
 6   num_late_payments_12m                14372 non-null  float64
 7   monthly_loan_repayment_debit_amount  14372 non-null  float64
 8   credit_score                         14372 non-null  float64
 9   credit_utilization_pct               14372 non-null  float64
 10  defaulted                            14372 non-null  float64
 11  annual_income_KES           

In [578]:
#dropping columns that are not to be used in modelling...
customers_data_agg=customers_data_agg.drop(columns=['customer_id','application_id','monthly_income_KES','monthly_loan_repayment_debit_amount'])

In [579]:
customers_data_agg.head()

,age,loan_amount_KES,num_existing_loans,num_late_payments_12m,credit_score,credit_utilization_pct,defaulted,annual_income_KES,debt_to_income_ratio%,loan_to_income_ratio%,employment_type
0,35.0,73257.0,2.0,4.0,577.0,93.0,0.0,1233288.0,0.000000,5.939975,Business Owner
1,18.0,25878.0,3.0,6.0,383.0,60.0,1.0,677376.0,52.605938,3.820330,Unknown
2,43.0,206404.0,5.0,9.0,376.0,68.0,1.0,421956.0,0.000000,48.916001,Unknown
3,64.0,214247.0,2.0,6.0,544.0,36.0,0.0,335184.0,0.000000,63.919220,Unemployed
4,60.0,130731.0,4.0,7.0,365.0,50.0,1.0,309864.0,56.072341,42.189799,Casual Worker


In [580]:
#instanciating the model...
lr=LogisticRegression(random_state=42)

In [623]:
#assigning the x and y variables...
x=customers_data_agg[['age','loan_amount_KES','annual_income_KES','loan_to_income_ratio%','debt_to_income_ratio%','credit_score']]
y=customers_data_agg['defaulted']

In [624]:
#Resampling x and y due to class imbalance...
sm=SMOTE(random_state=42)
x_resampled,y_resampled=sm.fit_resample(x,y)

In [625]:
# instanciating StandardScaler...
sc=StandardScaler()

In [626]:
#train_test_split the data...
x_train,x_test,y_train,y_test=train_test_split(x_resampled,y_resampled,test_size=0.2,random_state=42,stratify=y_resampled)

In [627]:
#normalizing the data..
x_train_scaled=sc.fit_transform(x_train)
x_test_scaled=sc.fit_transform(x_test)

In [628]:
#fitting in the model..
lr.fit(x_train_scaled,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [629]:
#predicting the labels...
lr_preds=lr.predict(x_test_scaled)

In [630]:
# Evaluating the model...
rec_lr= recall_score(y_test,lr_preds)
pre_lr=precision_score(y_test,lr_preds)
f1_lr=f1_score(y_test,lr_preds)
acc_lr=accuracy_score(y_test,lr_preds)

# print the results...
results=pd.DataFrame( data={'Model_name': 'LogisticRegression','recall':rec_lr,'precision': pre_lr, 'f1': f1_lr, 'accuracy': acc_lr},index=[0] )

In [631]:
results

,Model_name,recall,precision,f1,accuracy
0,LogisticRegression,0.739594,0.681478,0.709348,0.696954


In [632]:
print( classification_report(y_test,lr_preds) )

              precision    recall  f1-score   support

         0.0       0.72      0.65      0.68      1970
         1.0       0.68      0.74      0.71      1970

    accuracy                           0.70      3940
   macro avg       0.70      0.70      0.70      3940
weighted avg       0.70      0.70      0.70      3940



**Training model 2: `RandomForestClassifier`**

In [633]:
rf=RandomForestClassifier(random_state=42)

In [634]:
rf.fit(x_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [635]:
rf_preds=rf.predict(x_test)

In [636]:
# Evaluating the model...
rec_rf= recall_score(y_test,rf_preds)
pre_rf=precision_score(y_test,rf_preds)
f1_rf=f1_score(y_test,rf_preds)
acc_rf=accuracy_score(y_test,rf_preds)

# print the results...
rf_results=pd.DataFrame( data={'Model_name': 'RandomForestClassifier','recall':rec_rf,'precision': pre_rf, 'f1': f1_rf, 'accuracy': acc_rf},index=[1] )
results=pd.concat([results,rf_results])
results

,Model_name,recall,precision,f1,accuracy
0,LogisticRegression,0.739594,0.681478,0.709348,0.696954
1,RandomForestClassifier,0.773096,0.739320,0.755831,0.750254


In [637]:
print( classification_report(y_test,rf_preds) )

              precision    recall  f1-score   support

         0.0       0.76      0.73      0.74      1970
         1.0       0.74      0.77      0.76      1970

    accuracy                           0.75      3940
   macro avg       0.75      0.75      0.75      3940
weighted avg       0.75      0.75      0.75      3940



**Training Model 3: `XGBClassifier`**

In [638]:
xgb=XGBClassifier(random_state=42)

In [639]:
xgb.fit(x_train,y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [640]:
xgb_preds=xgb.predict(x_test)

In [641]:
# Evaluating the model...
rec_xgb= recall_score(y_test,xgb_preds)
pre_xgb=precision_score(y_test,xgb_preds)
f1_xgb=f1_score(y_test,xgb_preds)
acc_xgb=accuracy_score(y_test,xgb_preds)

# print the results...
xgb_results=pd.DataFrame( data={'Model_name': ' XGBClassifier','recall':rec_xgb,'precision': pre_xgb, 'f1': f1_xgb, 'accuracy': acc_xgb},index=[2] )
results=pd.concat([results,xgb_results])
results

,Model_name,recall,precision,f1,accuracy
0,LogisticRegression,0.739594,0.681478,0.709348,0.696954
1,RandomForestClassifier,0.773096,0.739320,0.755831,0.750254
2,XGBClassifier,0.770558,0.750371,0.760331,0.757107


In [642]:
print( classification_report(y_test,xgb_preds) )

              precision    recall  f1-score   support

         0.0       0.76      0.74      0.75      1970
         1.0       0.75      0.77      0.76      1970

    accuracy                           0.76      3940
   macro avg       0.76      0.76      0.76      3940
weighted avg       0.76      0.76      0.76      3940



In [646]:
results=results.sort_values(by='f1',ascending=False)

#creating a csv file for the trained_models_performance....
#results.to_csv('trained_models_performance.csv')
results

,Model_name,recall,precision,f1,accuracy
2,XGBClassifier,0.770558,0.750371,0.760331,0.757107
1,RandomForestClassifier,0.773096,0.739320,0.755831,0.750254
0,LogisticRegression,0.739594,0.681478,0.709348,0.696954


**Based on the f1 scores; `XGBClassifier` becomes the champion model...**

In [697]:
importances=xgb.feature_importances_

In [698]:
features=x.columns
features=pd.DataFrame(features)
features.columns=['features']
features

,features
0,age
1,loan_amount_KES
2,annual_income_KES
3,loan_to_income_ratio%
4,debt_to_income_ratio%
5,credit_score


In [699]:
features['importances']=importances

In [700]:
features=features.sort_values(by='importances',ascending=False)
features=features.set_index('features')
features=features.reset_index()

In [702]:
#creating a csv file for the feature importance for the champion model....
features.to_csv('XGBClassifier_feature_importances.csv')
features

,features,importances
0,credit_score,0.383714
1,debt_to_income_ratio%,0.226180
2,age,0.133482
3,loan_amount_KES,0.085994
4,annual_income_KES,0.085583
5,loan_to_income_ratio%,0.085046


In [703]:
#saving the model...
import pickle

filename = 'xgb_credit_finalized_model.pkl'

# Save the model 
with open(filename, 'wb') as file:
    pickle.dump(xgb, file)